# 🏆 Parameter Golf: God-Tier Hymba-7 Smoke Test
This notebook validates the Hymba architecture on a T4 GPU before deploying to 8xH100.

In [ ]:
# Step 1: Verify GPU
!nvidia-smi | head -12

In [ ]:
# Step 2: Install dependencies
!pip install -q torch sentencepiece zstandard numpy
!pip install -q mamba-ssm causal-conv1d
print('\n=== Dependencies installed ===')

In [ ]:
# Step 3: Clone the repo and download data
!git clone https://github.com/openai/parameter-golf.git /content/parameter-golf 2>&1 | tail -3
import os
os.chdir('/content/parameter-golf')
!python data/download.py 2>&1 | tail -5
print('\n=== Repo cloned and data downloaded ===')

In [ ]:
# Step 4: Upload hymba_train_gpt.py from GitHub fork
# We'll download it directly from the user's fork
!curl -sL https://raw.githubusercontent.com/Prush69/parameter-golf/main/hymba_train_gpt.py -o /content/parameter-golf/hymba_train_gpt.py 2>&1 || echo 'Fork not ready, will write inline'
!wc -l /content/parameter-golf/hymba_train_gpt.py

In [ ]:
# Step 5: Quick syntax check
!python -m py_compile hymba_train_gpt.py && echo '=== Syntax OK ===' || echo '=== SYNTAX ERROR ==='

In [ ]:
# Step 6: 100-step Shape Smoke Test (single GPU, tiny batch)
import os
os.environ['ITERATIONS'] = '100'
os.environ['WARMDOWN_ITERS'] = '0'
os.environ['WARMUP_STEPS'] = '2'
os.environ['TTT_ENABLED'] = '0'
os.environ['VAL_LOSS_EVERY'] = '50'
os.environ['VAL_BATCH_SIZE'] = '4096'
os.environ['TRAIN_BATCH_TOKENS'] = '4096'
os.environ['TRAIN_SEQ_LEN'] = '256'
os.environ['EVAL_BATCH_SEQS'] = '4'
os.environ['MAX_WALLCLOCK_SECONDS'] = '300'
os.environ['SWA_ENABLED'] = '0'
os.environ['EMA_ENABLED'] = '0'
os.environ['QAT_START_FRAC'] = '0'
os.environ['TRAIN_LOG_EVERY'] = '10'

!python hymba_train_gpt.py

In [ ]:
# Step 7: Artifact Size Check
import os
model_size = os.path.getsize('final_model.int8.ptz') if os.path.exists('final_model.int8.ptz') else 0
code_size = os.path.getsize('hymba_train_gpt.py')
total = model_size + code_size
limit = 16_777_216
print(f'Model artifact:  {model_size:>10,} bytes')
print(f'Code size:       {code_size:>10,} bytes')
print(f'Total:           {total:>10,} bytes')
print(f'Budget:          {limit:>10,} bytes')
print(f'Remaining:       {limit - total:>10,} bytes')
print(f'\n{"PASS" if total < limit else "FAIL"}: {"Under" if total < limit else "OVER"} 16MB limit')